In [1]:
import wbgapi as wb
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

## 1. PIB per capita e população (World Bank)

In [2]:
paises_codigos = {
    "Angola": "AGO",
    "Cabo Verde": "CPV",
    "Guiné-Bissau": "GNB",
    "Moçambique": "MOZ",
    "São Tomé e Príncipe": "STP",
}

# NY.GDP.PCAP.PP.CD = PIB per capita 
# SP.POP.TOTL       = população total
codigos_indicadores = {
    'NY.GDP.PCAP.PP.CD': 'pib_per_capita',
    'SP.POP.TOTL': 'populacao',
}

dados_paises = {}
for nome_pt in paises_codigos:
    dados_paises[nome_pt] = {}

for codigo_indicador, nome_coluna in codigos_indicadores.items():
    bruto = wb.data.DataFrame(codigo_indicador,
                              economy=list(paises_codigos.values()),
                              time=2023)
    for nome_pt, codigo_pais in paises_codigos.items():
        dados_paises[nome_pt][nome_coluna] = bruto.loc[codigo_pais].values[0]

indicadores_paises = pd.DataFrame(dados_paises).T
indicadores_paises

,pib_per_capita,populacao
Angola,9549.793139,36749906.0
Cabo Verde,10250.595025,522331.0
Guiné-Bissau,2955.523455,2153339.0
Moçambique,1680.208734,33635160.0
São Tomé e Príncipe,6155.050420,230871.0


## 2. IDH (PNUD)

In [3]:
paises_pnud = {
    "Angola": "Angola",
    "Cabo Verde": "Cabo Verde",
    "Guiné-Bissau": "Guinea-Bissau",
    "Moçambique": "Mozambique",
    "São Tomé e Príncipe": "Sao Tome and Principe",
}

idh_bruto = pd.read_excel("../DadosBrutos/HDR25_Statistical_Annex_HDI_Table.xlsx",
                          sheet_name="Table 1. HDI",
                          header=None)

idh_por_pais = {}
for nome_pt, nome_en in paises_pnud.items():
    linha = idh_bruto[idh_bruto[1].astype(str).str.strip() == nome_en]
    idh_por_pais[nome_pt] = linha[2].values[0]

indicadores_paises['idh'] = pd.Series(idh_por_pais)
indicadores_paises

,pib_per_capita,populacao,idh
Angola,9549.793139,36749906.0,0.616
Cabo Verde,10250.595025,522331.0,0.668
Guiné-Bissau,2955.523455,2153339.0,0.514
Moçambique,1680.208734,33635160.0,0.493
São Tomé e Príncipe,6155.050420,230871.0,0.637


In [4]:
indicadores_paises.index.name = 'pais'
indicadores_paises.to_csv("indicadores_paises.csv", encoding='utf-8-sig')
print("Salvo:", indicadores_paises.shape)   

Salvo: (5, 3)


## 4. Similaridade entre países (KNN, K=1)


In [5]:
indicadores = pd.read_csv("indicadores_paises.csv", encoding='utf-8-sig')

indicadores['populacao_log'] = np.log10(indicadores['populacao'])

colunas_atributos = ['pib_per_capita', 'populacao_log', 'idh']

escalador = MinMaxScaler()
atributos = escalador.fit_transform(indicadores[colunas_atributos])

tabela_atributos = pd.DataFrame(atributos,
                                columns=colunas_atributos,
                                index=indicadores['pais'])
print(tabela_atributos.round(3))

                     pib_per_capita  populacao_log    idh
pais                                                     
Angola                        0.918          1.000  0.703
Cabo Verde                    1.000          0.161  1.000
Guiné-Bissau                  0.149          0.440  0.120
Moçambique                    0.000          0.983  0.000
São Tomé e Príncipe           0.522          0.000  0.823


In [6]:
# n_neighbors=2 porque o primeiro vizinho que o sklearn retorna
# é sempre o próprio país (distância 0); o segundo é o vizinho de verdade
modelo = NearestNeighbors(n_neighbors=2, metric='euclidean')
modelo.fit(atributos)

distancias, indices = modelo.kneighbors(atributos)

vizinho_de = {}
distancia_do_vizinho = {}

for i in range(len(indicadores)):
    pais = indicadores.loc[i, 'pais']
    indice_vizinho = indices[i][1]
    vizinho_de[pais] = indicadores.loc[indice_vizinho, 'pais']
    distancia_do_vizinho[pais] = distancias[i][1]

for pais in vizinho_de:
    print(pais, "->", vizinho_de[pais], round(distancia_do_vizinho[pais], 3));

Angola -> Cabo Verde 0.894
Cabo Verde -> São Tomé e Príncipe 0.534
Guiné-Bissau -> Moçambique 0.575
Moçambique -> Guiné-Bissau 0.575
São Tomé e Príncipe -> Cabo Verde 0.534
